# MedGemma 1.5 Knowledge Distillation (QLoRA) — Chest X-Ray Report Generation & RadGraph F1 Evaluation

| | |
|---|---|
| **Method** | Knowledge Distillation |
| **Teacher** | Google MedGemma 1.5 (4B) original model |
| **Student** | MedGemma 1.5 + QLoRA (4-bit quantization + LoRA fine-tuning) |
| **Dataset** | MIMIC-CXR (233 samples) |
| **Eval Metric** | RadGraph F1 Score |
| **GPU** | A100 / H100 |

---

## ⚠️ Environment Requirements

- **Python:** 3.10–3.12 (Colab default 3.12 works)
- **GPU:** A100 or H100 (distillation training is memory-intensive, 40 GB+ recommended)
- **Dependencies:** transformers 5.x, radgraph, bitsandbytes, peft, trl
- **HuggingFace:** MedGemma access permission + token required (see Step 2)
- **Google Drive:** Upload `mimic_eval_single_image_final_233.csv` before running

---

## 🚨 Before You Start

1. **Step 0:** Check Python version (3.10–3.12 all supported)
2. **Step 2:** ⚠️ Log in to HuggingFace **first** (MedGemma is a gated model)
3. **Time estimate:** Distillation training takes 2–4 hours (depends on GPU and number of training steps)

---

## 🎓 Knowledge Distillation — How It Works

**Roles:**
- **Teacher model:** Original MedGemma 1.5 — generates high-quality reference reports
- **Student model:** MedGemma 1.5 with 4-bit quantization + LoRA fine-tuning — learns to mimic the teacher
- **Distillation objective:** Student fits the teacher's output token-by-token using cross-entropy (CE) loss

**Advantages:**
- Student model is smaller and faster (4-bit quantized)
- Maintains high generation quality by learning from teacher outputs
- LoRA fine-tuning makes training efficient and memory-friendly

---

## Pipeline Overview

| Step | Description |
|------|-------------|
| 0 | Check Python version (3.10–3.12) |
| 1 | Install dependencies (transformers + radgraph + peft + trl + bitsandbytes) |
| 2 | Log in to HuggingFace (**required!**) |
| 3 | Mount Google Drive |
| 4 | Download MIMIC-CXR dataset via kagglehub |
| 5 | Align image paths in the 233-sample CSV |
| 6 | Generate target reports with the Teacher model (or use pre-generated reports from CSV) |
| 7 | Initialize Student model (4-bit quantization + LoRA) |
| 8 | Distillation training (Student learns from Teacher outputs) |
| 9 | Generate reports with the trained Student model |
| 10 | RadGraph F1 evaluation |

## Step 0: 安装 Python 3.10（必需！）

⚠️ **重要**：Colab 默认是 Python 3.12，但某些依赖需要 3.10 或 3.11。

运行后需要 **Restart Runtime**（Runtime → Restart Runtime）。

## Step 1: 安装依赖

⚠️ **前置条件**：确保已安装 Python 3.10（Step 0）并重启了 Runtime。

In [ ]:
import sys

print(f"当前 Python 版本: {sys.version}")

# 检查 Python 版本
py_major = sys.version_info.major
py_minor = sys.version_info.minor

if py_major == 3 and py_minor in [10, 11]:
    print(f"✅ Python 3.{py_minor} 符合要求")
elif py_major == 3 and py_minor == 12:
    print("\\n" + "="*70)
    print("ℹ️ 检测到 Python 3.12")
    print("\\ntransformers 5.x 和 radgraph 都支持 Python 3.12，可以直接使用！")
    print("\\n如果遇到兼容性问题，可以尝试：")
    print("1. 降级 transformers：pip install transformers==4.47.1")
    print("2. 或在 Colab 设置中选择旧版 runtime")
    print("="*70)
    print("\\n✅ 继续使用 Python 3.12（建议先尝试）")
else:
    print(f"\\n⚠️ 未知 Python 版本: {py_major}.{py_minor}")
    print("建议使用 Python 3.10-3.12")

print("\\n跳过 Python 安装，直接进入 Step 1")

当前 Python 版本: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
\n======================================================================
ℹ️ 检测到 Python 3.12
\ntransformers 5.x 和 radgraph 都支持 Python 3.12，可以直接使用！
\n如果遇到兼容性问题，可以尝试：
1. 降级 transformers：pip install transformers==4.47.1
2. 或在 Colab 设置中选择旧版 runtime
\n✅ 继续使用 Python 3.12（建议先尝试）
\n跳过 Python 安装，直接进入 Step 1


## Step 2: 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive 已挂载")

Mounted at /content/drive
✅ Google Drive 已挂载


In [ ]:
import sys

# 验证 Python 版本
py_version = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"当前 Python 版本: {py_version}")

if sys.version_info.major == 3 and sys.version_info.minor in [10, 11, 12]:
    print(f"✅ Python {py_version} 兼容")
else:
    print(f"\n⚠️ 警告：当前版本 {py_version} 未测试，推荐 3.10-3.12")

# 安装依赖（蒸馏需要 peft, trl, bitsandbytes）
print("\n正在安装依赖...")
!pip install -U -q transformers
!pip install -q radgraph
!pip install -q bitsandbytes  # 量化必需
!pip install -q peft  # LoRA 微调必需
!pip install -q trl  # 训练工具
print("\n✅ 依赖安装完成（包含 bitsandbytes, peft, trl）")

当前 Python 版本: 3.12
✅ Python 3.12 兼容

正在安装依赖...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 142.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.0/588.0 kB 36.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 4.6 MB/s eta 0:00:00

✅ 依赖安装完成（包含 bitsandbytes, peft, trl）


## Step 3: 下载 MIMIC-CXR 数据集

使用 kagglehub 下载 MIMIC-CXR 数据集（~18GB）。首次运行需要 5-10 分钟。

In [ ]:
import kagglehub
import os

print("🚀 开始下载 MIMIC-CXR 数据集（~18GB，首次运行需要 5-10 分钟）...")

# kagglehub 会自动缓存，第二次运行会很快
dataset_path = kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset")

print(f"✅ 数据集下载完成！")
print(f"📂 存储路径: {dataset_path}")

# 验证文件结构
print("\n--- 文件夹结构预览 ---")
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level > 1: break  # 只显示前 2 层

🚀 开始下载 MIMIC-CXR 数据集（~18GB，首次运行需要 5-10 分钟）...


100%|██████████| 16.5G/16.5G [13:30<00:00, 21.9MB/s]

Extracting files...


✅ 数据集下载完成！
📂 存储路径: /root/.cache/kagglehub/datasets/simhadrisadaram/mimic-cxr-dataset/versions/2

--- 文件夹结构预览 ---
2/
  official_data_iccv_final/
    files/


## Step 4: 加载 233 CSV 并对齐路径

读取 `mimic_eval_single_image_final_233.csv`，并将 CSV 中的路径对齐到 kagglehub 下载的实际路径。

In [ ]:
import pandas as pd
import os

# ✅ 使用 233 CSV
csv_path = "/content/drive/MyDrive/medgamma/mimic_eval_single_image_final_233.csv"

if not os.path.exists(csv_path):
    print(f"❌ 找不到 CSV 文件: {csv_path}")
    print("\n请确保 233 CSV 已上传到 Google Drive 的 medgamma 文件夹中！")
    raise FileNotFoundError(csv_path)

print(f"📂 读取 CSV: {csv_path}")
df = pd.read_csv(csv_path)
print(f"✅ 加载成功！共 {len(df)} 条数据")
print(f"\n列名: {list(df.columns)}")

# ==========================================
# 路径对齐函数
# ==========================================
dataset_root = f"{dataset_path}/official_data_iccv_final"

def fix_image_path(path_in_csv):
    """
    将 233 CSV 中的路径对齐到 kagglehub 下载的实际路径
    CSV 格式: /kaggle/input/mimic-cxr-dataset/official_data_iccv_final/files/...
    实际格式: {dataset_root}/files/...
    """
    if pd.isna(path_in_csv):
        return None

    path_str = str(path_in_csv).strip()

    # 提取从 files/ 开始的相对路径
    if 'files/' in path_str:
        relative_part = path_str.split('files/', 1)[1]
        full_path = os.path.join(dataset_root, 'files', relative_part)
        return full_path if os.path.exists(full_path) else None

    return None

# 应用路径修正
print("\n🔄 正在对齐图片路径...")
if 'Image_Path' in df.columns:
    df['Image_Path'] = df['Image_Path'].apply(fix_image_path)
    # 验证
    valid_count = df['Image_Path'].notna().sum()
    print(f"✅ 路径对齐完成！有效路径: {valid_count}/{len(df)}")

    # 显示第一个有效路径
    first_valid = df['Image_Path'].dropna().iloc[0]
    print(f"示例路径: {first_valid}")
    print(f"文件存在: {os.path.exists(first_valid)}")
else:
    print("❌ CSV 中没有 Image_Path 列！")

📂 读取 CSV: /content/drive/MyDrive/medgamma/mimic_eval_single_image_final_233.csv
✅ 加载成功！共 233 条数据

列名: ['subject_id', 'View', 'Image_Path', 'Ground_Truth', 'Generated_Report']

🔄 正在对齐图片路径...
✅ 路径对齐完成！有效路径: 233/233
示例路径: /root/.cache/kagglehub/datasets/simhadrisadaram/mimic-cxr-dataset/versions/2/official_data_iccv_final/files/p10/p10075925/s51010496/2d783c8a-492984b7-28aaf571-bfc30156-61ab26f6.jpg
文件存在: True


## Step 6: 使用 Teacher 模型生成目标报告

**说明**: 如果 CSV 中已有 `Generated_Report` 列且质量较好，可以直接使用。

否则需要先加载原始 MedGemma 作为 Teacher 生成目标报告。

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    # 使用你已设置的 Secret 名称（zhuxirui11）
    hf_token = userdata.get('zhuxirui11')
    login(token=hf_token)
    print("✅ HuggingFace 登录成功！")
except Exception as e:
    print("❌ 登录失败！")
    print(f"错误信息: {e}")
    print("\n请确认：")
    print("1. 已在 https://huggingface.co/google/medgemma-1.5-4b-it 申请访问权限")
    print("2. 已在 Colab 左侧 🔑 Secrets 中添加 zhuxirui11")
    print("3. 已勾选 'Notebook access'（开关是蓝色的）")
    raise

✅ HuggingFace 登录成功！


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# ===== Teacher 模型（原始 MedGemma）=====
model_id = "google/medgemma-1.5-4b-it"

print(f"🤖 正在加载 Teacher 模型: {model_id}...")
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "google/medgemma-1.5-4b-it"

print(f"🤖 正在加载模型: {model_id}...")
print("⚠️ 首次加载需要下载 ~8GB 权重，请耐心等待...\n")

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,  # A100 支持 BF16
    device_map="auto",
    trust_remote_code=True
)

print(f"✅ 模型加载成功！")
print(f"Device: {model.device}")
if torch.cuda.is_available():
    mem_gb = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"GPU 显存占用: {mem_gb:.2f} GB")

🤖 正在加载 Teacher 模型: google/medgemma-1.5-4b-it...
🤖 正在加载模型: google/medgemma-1.5-4b-it...
⚠️ 首次加载需要下载 ~8GB 权重，请耐心等待...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

✅ 模型加载成功！
Device: cuda:0
GPU 显存占用: 8.01 GB


## Step 6: 批量生成报告（233 samples）

对 233 张胸部 X 光图片生成放射学报告。

In [ ]:
import re
from PIL import Image
from tqdm import tqdm

def generate_one_report(image_path, view_position):
    """
    为单张图片生成放射学报告
    """
    # 提示词
    prompt_text = (
        f"You are an expert radiologist. Describe this {view_position} view chest X-ray. "
        "Provide a concise report consisting of Findings and Impression. "
        "Focus on the heart, lungs, mediastinum, pleural space, and bones. "
        "Do NOT use bullet points, asterisks, or section headers. "
        "Do NOT include disclaimers or 'AI' warnings. "
        "Output pure medical text only."
    )

    try:
        pil_image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return f"ERROR_IMAGE_LOAD: {e}"

    # 构建输入
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    # 推理
    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )
        generation = generation[0][input_len:]  # 裁剪掉 prompt

    # ✅ 解码并强力清洗，去除格式标记
    raw_text = processor.decode(generation, skip_special_tokens=True)
    clean_text = raw_text.replace("Findings:", "").replace("Impression:", "")
    clean_text = clean_text.replace("**", "").replace("*", "")  # 去除 markdown
    clean_text = clean_text.replace("###", "").replace("##", "").replace("#", "")
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    return clean_text

# ==========================================
# 批量生成
# ==========================================
print(f"\n🚀 开始批量生成报告（共 {len(df)} 条）...\n")

results = []
skipped_count = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="生成报告"):
    try:
        # ✅ 使用 Image_Path 列
        img_path = row.get('Image_Path')
        view = row.get('View', 'PA')

        # 验证路径
        if not img_path or not os.path.exists(img_path):
            skipped_count += 1
            continue

        # 生成报告
        generated_report = generate_one_report(img_path, view)

        # 检查错误
        if "ERROR_IMAGE_LOAD" in generated_report:
            skipped_count += 1
            continue

        # ✅ Ground_Truth 列
        gt_col = 'Ground_Truth' if 'Ground_Truth' in df.columns else 'text'
        ground_truth = str(row.get(gt_col, '')).strip()

        # 保存结果
        results.append({
            "subject_id": row.get('subject_id', idx),
            "View": view,
            "Image_Path": img_path,
            "Ground_Truth": ground_truth,
            "Generated_Report": generated_report
        })

    except Exception as e:
        print(f"\n❌ Error at index {idx}: {e}")
        skipped_count += 1
        continue

# 保存结果
df_results = pd.DataFrame(results)
output_path = "/content/drive/MyDrive/medgamma/medgemma_distilled_reports_233.csv"
df_results.to_csv(output_path, index=False)

print(f"\n✅ 报告生成完成！")
print(f"成功生成: {len(results)} 条")
print(f"跳过: {skipped_count} 条")
print(f"结果已保存至: {output_path}")

# 预览前 3 条
print("\n--- 前 3 条报告预览 ---")
for i in range(min(3, len(df_results))):
    print(f"\n[{i+1}] Subject: {df_results.iloc[i]['subject_id']}")
    print(f"Ground Truth: {df_results.iloc[i]['Ground_Truth'][:100]}...")
    print(f"Generated: {df_results.iloc[i]['Generated_Report'][:100]}...")


🚀 开始批量生成报告（共 233 条）...



生成报告: 100%|██████████| 233/233 [15:11<00:00,  3.91s/it]



✅ 报告生成完成！
成功生成: 233 条
跳过: 0 条
结果已保存至: /content/drive/MyDrive/medgamma/medgemma_distilled_reports_233.csv

--- 前 3 条报告预览 ---

[1] Subject: 10075925
Ground Truth: Mild pulmonary vascular congestion with mild to moderate interstitial pulmonary edema are new compar...
Generated: The heart is enlarged. There is mild pulmonary edema. The mediastinum is unremarkable. The pleural s...

[2] Subject: 10174198
Ground Truth: Lungs are clear without consolidation, effusion, or pneumothorax.  The cardiomediastinal silhouette ...
Generated: The heart size is normal. The mediastinal contours are unremarkable. The lungs are clear without foc...

[3] Subject: 10199765
Ground Truth: Subtle patchy opacity along the left heart border on the frontal view, not substantiated on the late...
Generated: The heart size is mildly enlarged. The mediastinal contours are unremarkable. The lungs are clear wi...


## Step 7: 保存 Teacher 输出

将 Teacher 模型生成的报告保存，作为 Student 的学习目标。

In [ ]:
# 保存 Teacher 输出到 CSV
teacher_csv_path = "/content/drive/MyDrive/medgamma/teacher_reports_233.csv"
df_results.to_csv(teacher_csv_path, index=False)
print(f"✅ Teacher 报告已保存至: {teacher_csv_path}")

# 释放 Teacher 模型显存
import gc
del model, processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Teacher 模型已卸载")

✅ Teacher 报告已保存至: /content/drive/MyDrive/medgamma/teacher_reports_233.csv
✅ Teacher 模型已卸载


## Step 8: 初始化 Student 模型（4-bit + LoRA）

**配置**:
- **量化**: 4-bit (bitsandbytes)
- **微调**: LoRA (Parameter-Efficient Fine-Tuning)
- **训练目标**: 学习 Teacher 的输出

In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "google/medgemma-1.5-4b-it"

print(f"🤖 正在加载 Student 模型: {model_id} (4-bit + LoRA)...\n")

# 4-bit 量化配置
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 加载 processor
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

# 加载 Student 模型（4-bit 量化）
student_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

# 准备模型用于 k-bit 训练
student_model = prepare_model_for_kbit_training(student_model)

# LoRA 配置
lora_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # 应用 LoRA 的模块
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 应用 LoRA
student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()

print(f"\n✅ Student 模型（4-bit + LoRA）初始化完成！")
if torch.cuda.is_available():
    mem_gb = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"GPU 显存占用: {mem_gb:.2f} GB")

🤖 正在加载 Student 模型: google/medgemma-1.5-4b-it (4-bit + LoRA)...



Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

trainable params: 6,447,104 || all params: 4,306,526,576 || trainable%: 0.1497

✅ Student 模型（4-bit + LoRA）初始化完成！
GPU 显存占用: 4.32 GB


In [ ]:
# 检查可训练参数
trainable = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in student_model.parameters())
print(f"可训练参数: {trainable:,} / {total:,}")

可训练参数: 6,447,104 / 2,496,670,064


## Step 9: 蒸馏训练

**训练配置**:
- **Epochs**: 3-5
- **Batch size**: 1-2（取决于显存）
- **Learning rate**: 2e-4
- **Loss**: Cross Entropy（逐 token 拟合 Teacher 输出）

⚠️ **预计时间**: 2-4 小时

In [ ]:
import pandas as pd

df_results = pd.read_csv("/content/drive/MyDrive/medgamma/medgemma_distilled_reports_233.csv")
print(f"✅ 加载成功: {len(df_results)} 条")
print(df_results.columns.tolist())

✅ 加载成功: 233 条
['subject_id', 'View', 'Image_Path', 'Ground_Truth', 'Generated_Report']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print(df_results.iloc[0]["Generated_Report"])

The heart is enlarged. There is mild pulmonary edema. The mediastinum is unremarkable. The pleural spaces are clear. The bones are unremarkable. Cardiomegaly with mild pulmonary edema.


In [ ]:
# ==========================================
# 方案 2: 固定长度方案（修复版）
# ==========================================

from transformers import Trainer, TrainingArguments, TrainerCallback
from torch.utils.data import Dataset
import torch
from PIL import Image

class DistillationDataset(Dataset):
    def __init__(self, df, processor, image_root, max_length=512):
        self.df = df
        self.processor = processor
        self.image_root = image_root
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["Image_Path"]
        teacher_output = row["Generated_Report"]

        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new("RGB", (224, 224), (255, 255, 255))

        # ⭐ 修复：把 prompt + teacher_output 拼在一起作为完整序列
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Describe this chest X-ray in detail."}]}]
        prompt = self.processor.apply_chat_template(messages, add_generation_prompt=True)
        full_text = prompt + teacher_output  # ⭐ 拼接

        inputs = self.processor(
            text=full_text,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )

        # ⭐ 修复：labels 和 input_ids 用同一个序列，只让模型学 teacher_output 部分
        labels = inputs["input_ids"].clone()

        # prompt 部分设为 -100（不计算 loss）
        prompt_tokens = self.processor.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length
        )
        prompt_len = prompt_tokens["input_ids"].shape[1]
        labels[0, :prompt_len] = -100  # 忽略 prompt 部分
        labels[labels == self.processor.tokenizer.pad_token_id] = -100  # 忽略 padding

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "pixel_values": inputs["pixel_values"].squeeze(),
            "labels": labels.squeeze()
        }


class CustomDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        batch = {
            "input_ids": torch.stack([f["input_ids"] for f in features]),
            "attention_mask": torch.stack([f["attention_mask"] for f in features]),
            "pixel_values": torch.stack([f["pixel_values"] for f in features]),
            "labels": torch.stack([f["labels"] for f in features]),
        }
        batch_size, seq_length = batch["input_ids"].shape
        batch["token_type_ids"] = torch.zeros((batch_size, seq_length), dtype=torch.long)
        return batch


class SaveBestModelCallback(TrainerCallback):
    def __init__(self, save_path, processor):
        self.save_path = save_path
        self.processor = processor
        self.best_loss = float("inf")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        current_loss = logs.get("loss")
        if current_loss is not None and current_loss < self.best_loss:
            self.best_loss = current_loss
            kwargs["model"].save_pretrained(self.save_path)
            self.processor.save_pretrained(self.save_path)
            print(f"\n💾 Best model saved (loss={self.best_loss:.4f}) → {self.save_path}")


# ==========================================
# 路径定义
# ==========================================

BEST_MODEL_PATH  = "/content/drive/MyDrive/medgamma/distilled_best_model_v2"
FINAL_MODEL_PATH = "/content/drive/MyDrive/medgamma/distilled_final_model_v2"

# ==========================================
# 训练
# ==========================================

train_dataset = DistillationDataset(
    df_results,  # ⭐ 改这里
    processor,
    dataset_root,
    max_length=512
)
data_collator = CustomDataCollator(processor)

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/medgamma/distillation_checkpoints_v2",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,        # ⭐ 从 2e-4 降到 2e-5
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=3,
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=1.0,         # ⭐ 梯度裁剪
    warmup_steps=20,           # ⭐ warmup
)

trainer = Trainer(
    model=student_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    callbacks=[SaveBestModelCallback(BEST_MODEL_PATH, processor)],
)

# ⭐ 训练前验证前几条 loss 是否正常（应该从 >1 开始下降）
print("🔍 训练前验证前5步 loss...")
print("\n🚀 开始蒸馏训练...\n")
trainer.train()
print("\n✅ 蒸馏训练完成！")

student_model.save_pretrained(FINAL_MODEL_PATH)
processor.save_pretrained(FINAL_MODEL_PATH)
print(f"✅ Final model saved → {FINAL_MODEL_PATH}")
print(f"✅ Best  model saved → {BEST_MODEL_PATH}")

🔍 训练前验证前5步 loss...

🚀 开始蒸馏训练...



/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
# ==========================================
# 训练后快速验证：检查模型是否崩了
# ==========================================

from PIL import Image
import torch

student_model.eval()

# 用一张真实的测试图片
test_image_path = df_results.iloc[0]["Image_Path"]
try:
    image = Image.open(test_image_path).convert("RGB")
except:
    image = Image.new("RGB", (224, 224), (128, 128, 128))

messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": "Describe this chest X-ray in detail."}
]}]
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=image, return_tensors="pt").to(student_model.device)

with torch.no_grad():
    outputs = student_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.0  # 不加任何惩罚，看模型原始输出
    )

generated = processor.decode(outputs[0], skip_special_tokens=True)
# 只取生成部分，去掉 prompt
response = generated[len(prompt):]

print("=" * 50)
print("生成内容:")
print(response)
print("=" * 50)

# 自动检查是否崩了
words = response.strip().split()
if len(words) > 5:
    unique_ratio = len(set(words)) / len(words)
    print(f"\n词汇多样性: {unique_ratio:.2f}  (>0.3 正常，<0.1 崩了)")
    if unique_ratio < 0.1:
        print("❌ 模型崩了，存在大量重复 token")
    elif unique_ratio < 0.3:
        print("⚠️  输出有些重复，但还能用")
    else:
        print("✅ 输出正常")
else:
    print("❌ 输出太短，模型可能没有正常生成")

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


生成内容:
egaly). The heart silhouette seems to occupy a significant portion of the thoracic cavity. The lung fields show some increased interstitial markings, which could suggest pulmonary edema or other interstitial lung disease. There is also some patchy opacity in the right lower lung field, which could represent pneumonia or other consolidation. The mediastinum appears normal in width. The bony structures of the ribs and clavicles are visible. The patient's position is slightly rotated.

It is important to note that this is just a single image and a full clinical evaluation is needed to determine the significance of these findings. Further imaging or clinical information may be required for a definitive diagnosis.

词汇多样性: 0.73  (>0.3 正常，<0.1 崩了)
✅ 输出正常


## Step 7: RadGraph F1 评估

使用 RadGraph 指标评估生成报告的质量。

**指标说明：**
- **RG_E**: Entity F1（实体匹配）
- **RG_ER**: Entity + Relation F1（实体+关系，论文常用指标）
- **RG_ER_bar**: Complete Match F1（完全匹配）

In [ ]:
# ⚠️ RadGraph 兼容性修复（必须先运行这个 cell！）
# 修复 transformers 5.x 与 radgraph 的兼容性问题

from transformers import BertTokenizer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

fixed_methods = []

# 1. 添加 encode_plus 方法
if not hasattr(BertTokenizer, 'encode_plus'):
    def encode_plus_wrapper(self, text, *args, **kwargs):
        return self(text, *args, **kwargs)
    BertTokenizer.encode_plus = encode_plus_wrapper
    fixed_methods.append('encode_plus')

# 2. 添加 build_inputs_with_special_tokens 方法
if not hasattr(BertTokenizer, 'build_inputs_with_special_tokens'):
    def build_inputs_with_special_tokens_wrapper(self, token_ids_0, token_ids_1=None):
        # BERT: [CLS] + tokens_0 + [SEP] + tokens_1 + [SEP]
        if token_ids_1 is None:
            return [self.cls_token_id] + token_ids_0 + [self.sep_token_id]
        return [self.cls_token_id] + token_ids_0 + [self.sep_token_id] + token_ids_1 + [self.sep_token_id]
    BertTokenizer.build_inputs_with_special_tokens = build_inputs_with_special_tokens_wrapper
    fixed_methods.append('build_inputs_with_special_tokens')

# 3. 添加 get_special_tokens_mask 方法（可能也会需要）
if not hasattr(BertTokenizer, 'get_special_tokens_mask'):
    def get_special_tokens_mask_wrapper(
        self, token_ids_0, token_ids_1=None, already_has_special_tokens=False
    ):
        if already_has_special_tokens:
            return super(BertTokenizer, self).get_special_tokens_mask(
                token_ids_0=token_ids_0,
                token_ids_1=token_ids_1,
                already_has_special_tokens=True,
            )
        if token_ids_1 is None:
            return [1] + ([0] * len(token_ids_0)) + [1]
        return [1] + ([0] * len(token_ids_0)) + [1] + ([0] * len(token_ids_1)) + [1]
    BertTokenizer.get_special_tokens_mask = get_special_tokens_mask_wrapper
    fixed_methods.append('get_special_tokens_mask')

if fixed_methods:
    print(f"✅ RadGraph 兼容性修复已应用: {', '.join(fixed_methods)}")
else:
    print("✅ BertTokenizer 已有所有必需方法")

✅ RadGraph 兼容性修复已应用: encode_plus, build_inputs_with_special_tokens


In [ ]:
import torch

print("=" * 60)
print("📊 GPU 显存使用情况")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    current_gb = torch.cuda.memory_allocated(0) / (1024**3)
    peak_gb = torch.cuda.max_memory_allocated(0) / (1024**3)

    print(f"\nGPU: {gpu_name}")
    print(f"总显存: {total_gb:.2f} GB")
    print(f"当前使用: {current_gb:.2f} GB")
    print(f"峰值使用: {peak_gb:.2f} GB  ← 这是模型加载时的最大显存")
    print(f"剩余: {total_gb - current_gb:.2f} GB")
else:
    print("CPU 模式")

print("=" * 60)


# ==========================================
# 第 2 步: 准备评估数据
# ==========================================

import pandas as pd

print("\n📁 准备评估数据...")

# 选择一个方法加载数据:

# 方法 A: 从保存的文件加载
# df_sub = pd.read_csv("/content/drive/MyDrive/medgamma/generated_reports.csv")

# 方法 B: 如果你有 df_results 变量
df_sub = df_results.copy()

# 方法 C: 如果你有其他变量名
# df_sub = your_dataframe_name.copy()

# 验证数据
print(f"✅ 数据加载成功: {len(df_sub)} 条")
print(f"列名: {df_sub.columns.tolist()}")


# ==========================================
# 第 3 步: RadGraph 评估
# ==========================================

from radgraph import F1RadGraph

print("\n🔍 开始 RadGraph F1 评估...")

# 准备数据
refs = df_sub["Ground_Truth"].fillna("").tolist()
hyps = df_sub["Generated_Report"].fillna("").tolist()

# 过滤空报告
valid_pairs = [(h, r) for h, r in zip(hyps, refs) if h and r and len(h.strip()) > 0]
print(f"有效样本: {len(valid_pairs)}")

hyps_clean, refs_clean = zip(*valid_pairs)

# 计算
print("计算中...")
f1radgraph = F1RadGraph(reward_level="all")
results = f1radgraph(hyps=list(hyps_clean), refs=list(refs_clean))

# 结果
avg_scores = results[0]
simple_f1 = float(avg_scores[0])
partial_f1 = float(avg_scores[1])
complete_f1 = float(avg_scores[2])

# 显示
print("\n" + "=" * 60)
print("RadGraph F1 评估结果")
print("-" * 60)
print(f"RG_E:      {complete_f1*100:.2f}%")
print(f"RG_ER:     {partial_f1*100:.2f}%  ← 论文常用")
print(f"RG_ER_bar: {simple_f1*100:.2f}%")
print(f"\nGPU 峰值: {peak_gb:.2f} GB")
print("=" * 60)

print("\n✅ 完成!")

📊 GPU 显存使用情况

GPU: NVIDIA A100-SXM4-80GB
总显存: 79.25 GB
当前使用: 4.39 GB
峰值使用: 8.13 GB  ← 这是模型加载时的最大显存
剩余: 74.86 GB

📁 准备评估数据...
✅ 数据加载成功: 233 条
列名: ['subject_id', 'View', 'Image_Path', 'Ground_Truth', 'Generated_Report']

🔍 开始 RadGraph F1 评估...
有效样本: 233
计算中...
Using device: cuda:0
model_type not provided, defaulting to radgraph-xl


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


radgraph-xl.tar.gz:   0%|          | 0.00/416M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/radgraph/radgraph.py:105: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=model_dir)


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/bert/tokenization_bert.py:104: DeprecationWarning: Deprecated in 0.9.0: WordPiece.__init__ will not create from files anymore, try `WordPiece.from_file` instead
  self._tokenizer = Tokenizer(WordPiece(self._vocab, unk_token=str(unk_token)))



RadGraph F1 评估结果
------------------------------------------------------------
RG_E:      27.18%
RG_ER:     34.76%  ← 论文常用
RG_ER_bar: 36.54%

GPU 峰值: 8.13 GB

✅ 完成!


In [ ]:
# ==========================================
# Step 8.1: 使用 Student 模型生成目标报告 (v2 - Best Model, W4A8)
# ==========================================

!pip install torchao -q  # ⭐ 新增

import re
from PIL import Image
from tqdm import tqdm
import pandas as pd
import torch
import os
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig
from torchao.quantization import quantize_, int8_dynamic_activation_int4_weight  # ⭐ 新增

# ==========================================
# 加载 Best Student 模型
# ==========================================

BEST_MODEL_PATH = "/content/drive/MyDrive/medgamma/distilled_best_model_v2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    "google/medgemma-1.5-4b-it",
    quantization_config=bnb_config,
    device_map="auto"
)
student_model = PeftModel.from_pretrained(base_model, BEST_MODEL_PATH)
processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it")

# ⭐ 新增：推理阶段激活值压到 int8 → W4A8
quantize_(student_model, int8_dynamic_activation_int4_weight())

print(f"✅ Best Student 模型加载成功（W4A8）")
print(f"显存占用: {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")

# 临时将 student_model 赋值给 model
original_model = None
if 'model' in globals():
    original_model = globals()['model']
globals()['model'] = student_model

print(f"\n🚀 开始批量生成报告（共 {len(df)} 条）...\n")

student_results = []
skipped_count_student = 0

def generate_one_report_for_student(image_path, view_position):
    prompt_text = (
        f"You are an expert radiologist. Describe this {view_position} view chest X-ray. "
        "Provide a concise report consisting of Findings and Impression. "
        "Focus on the heart, lungs, mediastinum, pleural space, and bones. "
        "Do NOT use bullet points, asterisks, or section headers. "
        "Do NOT include disclaimers or 'AI' warnings. "
        "Output pure medical text only."
    )

    try:
        pil_image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return f"ERROR_IMAGE_LOAD: {e}"

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )
        generation = generation[0][input_len:]

    raw_text = processor.decode(generation, skip_special_tokens=True)
    clean_text = raw_text.replace("Findings:", "").replace("Impression:", "")
    clean_text = clean_text.replace("**", "").replace("*", "")
    clean_text = clean_text.replace("###", "").replace("##", "").replace("#", "")
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    return clean_text


for idx, row in tqdm(df.iterrows(), total=len(df), desc="生成 Student 报告"):
    try:
        img_path = row.get('Image_Path')
        view = row.get('View', 'PA')

        if not img_path or not os.path.exists(img_path):
            skipped_count_student += 1
            continue

        generated_report_student = generate_one_report_for_student(img_path, view)

        if "ERROR_IMAGE_LOAD" in generated_report_student:
            skipped_count_student += 1
            continue

        gt_col = 'Ground_Truth' if 'Ground_Truth' in df.columns else 'text'
        ground_truth = str(row.get(gt_col, '')).strip()

        student_results.append({
            "subject_id": row.get('subject_id', idx),
            "View": view,
            "Image_Path": img_path,
            "Ground_Truth": ground_truth,
            "Generated_Report": generated_report_student
        })

    except Exception as e:
        print(f"\n❌ Error at index {idx}: {e}")
        skipped_count_student += 1
        continue


df_student_generated_reports = pd.DataFrame(student_results)
output_path_student = "/content/drive/MyDrive/medgamma/student_distilled_reports_v2_233.csv"
df_student_generated_reports.to_csv(output_path_student, index=False)

print(f"\n✅ Student 报告生成完成！")
print(f"成功生成: {len(student_results)} 条")
print(f"跳过: {skipped_count_student} 条")
print(f"结果已保存至: {output_path_student}")
print(f"最终显存占用: {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")

# 恢复原始 model 变量
if original_model is not None:
    globals()['model'] = original_model

# 清理显存
del student_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Student 模型已卸载")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

✅ Best Student 模型加载成功（W4A8）
显存占用: 8.56 GB

🚀 开始批量生成报告（共 233 条）...



生成 Student 报告: 100%|██████████| 233/233 [46:55<00:00, 12.08s/it]



✅ Student 报告生成完成！
成功生成: 233 条
跳过: 0 条
结果已保存至: /content/drive/MyDrive/medgamma/student_distilled_reports_v2_233.csv
最终显存占用: 8.56 GB
✅ Student 模型已卸载


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# ⭐ 直接加载，不保存 base_model 变量
student_model = AutoModelForCausalLM.from_pretrained(
    "google/medgemma-1.5-4b-it",
    quantization_config=bnb_config,
    device_map="auto"
)
student_model = PeftModel.from_pretrained(
    student_model,
    "/content/drive/MyDrive/medgamma/distilled_best_model_v2"
)
processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it")

print(f"显存占用: {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

显存占用: 11.61 GB


In [ ]:
import torch
import pandas as pd
import gc
from radgraph import F1RadGraph

# ==========================================
# Step 1: Clean up duplicate model references
# ==========================================
if 'base_model' in globals():
    del base_model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"✅ base_model released: {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")

# ==========================================
# Step 2: GPU Memory Check
# ==========================================
print("=" * 60)
print("📊 GPU Memory Usage")
print("=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    torch.cuda.reset_peak_memory_stats()
    current_gb = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"GPU: {gpu_name}")
    print(f"Total VRAM:  {total_gb:.2f} GB")
    print(f"Before eval: {current_gb:.2f} GB")
    print(f"Available:   {total_gb - current_gb:.2f} GB")
else:
    print("CPU mode")
print("=" * 60)

# ==========================================
# Step 3: Load Evaluation Data
# ==========================================
print("\n📁 Loading Student model reports...")
df_sub = pd.read_csv("/content/drive/MyDrive/medgamma/student_distilled_reports_v2_233.csv")
print(f"✅ Loaded: {len(df_sub)} samples")

# ==========================================
# Step 4: RadGraph Evaluation
# ==========================================
print("\n🔍 Starting RadGraph F1 Evaluation (Student model)...")

refs = df_sub["Ground_Truth"].fillna("").tolist()
hyps = df_sub["Generated_Report"].fillna("").tolist()

valid_pairs = [(h, r) for h, r in zip(hyps, refs) if h and r and len(h.strip()) > 0]
print(f"Valid samples: {len(valid_pairs)}")
hyps_clean, refs_clean = zip(*valid_pairs)

print("Computing...")
f1radgraph = F1RadGraph(reward_level="all")
results = f1radgraph(hyps=list(hyps_clean), refs=list(refs_clean))

# Release RadGraph immediately
del f1radgraph
gc.collect()
torch.cuda.empty_cache()

after_gb = torch.cuda.memory_allocated(0) / (1024**3)
peak_gb  = torch.cuda.max_memory_allocated(0) / (1024**3)

avg_scores  = results[0]
simple_f1   = float(avg_scores[0])
partial_f1  = float(avg_scores[1])
complete_f1 = float(avg_scores[2])

print("\n" + "=" * 60)
print("RadGraph F1 Evaluation Results (Student Model v2)")
print("-" * 60)
print(f"RG_E:      {complete_f1*100:.2f}%")
print(f"RG_ER:     {partial_f1*100:.2f}%  ← commonly used in papers")
print(f"RG_ER_bar: {simple_f1*100:.2f}%")
print("-" * 60)
print(f"Student model VRAM: {after_gb:.2f} GB")
print(f"Peak VRAM:          {peak_gb:.2f} GB")
print("=" * 60)
print("\n✅ Done!")

✅ base_model released: 11.61 GB
📊 GPU Memory Usage
GPU: NVIDIA A100-SXM4-80GB
Total VRAM:  79.25 GB
Before eval: 11.61 GB
Available:   67.64 GB

📁 Loading Student model reports...
✅ Loaded: 233 samples

🔍 Starting RadGraph F1 Evaluation (Student model)...
Valid samples: 233
Computing...
Using device: cuda:0
model_type not provided, defaulting to radgraph-xl


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



RadGraph F1 Evaluation Results (Student Model v2)
------------------------------------------------------------
RG_E:      27.72%
RG_ER:     35.75%  ← commonly used in papers
RG_ER_bar: 37.14%
------------------------------------------------------------
Student model VRAM: 11.17 GB
Peak VRAM:          13.78 GB

✅ Done!


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # ⭐ 强制 RadGraph 用 CPU

import torch
import pandas as pd
from radgraph import F1RadGraph

# 重置显存统计
torch.cuda.reset_peak_memory_stats()

df_student_generated_reports = pd.read_csv(
    "/content/drive/MyDrive/medgamma/student_distilled_reports_v2_233.csv"
)
df_sub = df_student_generated_reports.copy()

refs = df_sub["Ground_Truth"].fillna("").tolist()
hyps = df_sub["Generated_Report"].fillna("").tolist()
valid_pairs = [(h, r) for h, r in zip(hyps, refs) if h and r and len(h.strip()) > 0]
hyps_clean, refs_clean = zip(*valid_pairs)

f1radgraph = F1RadGraph(reward_level="all")
results = f1radgraph(hyps=list(hyps_clean), refs=list(refs_clean))

avg_scores = results[0]
simple_f1   = float(avg_scores[0])
partial_f1  = float(avg_scores[1])
complete_f1 = float(avg_scores[2])

# 恢复 GPU 可见
os.environ.pop("CUDA_VISIBLE_DEVICES", None)

print("\n" + "=" * 60)
print("RadGraph F1 评估结果 (Student 模型 v2)")
print("-" * 60)
print(f"RG_E:      {complete_f1*100:.2f}%")
print(f"RG_ER:     {partial_f1*100:.2f}%  ← 论文常用")
print(f"RG_ER_bar: {simple_f1*100:.2f}%")
print("=" * 60)

Using device: cuda:0
model_type not provided, defaulting to radgraph-xl


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



RadGraph F1 评估结果 (Student 模型 v2)
------------------------------------------------------------
RG_E:      27.72%
RG_ER:     35.75%  ← 论文常用
RG_ER_bar: 37.14%


In [ ]:
import os
import torch
import pandas as pd
import transformers
from radgraph import F1RadGraph
from transformers import BertTokenizer, BertTokenizerFast, PreTrainedTokenizerBase

# ==========================================
# 🔧 Compatibility Patch
# ==========================================
print(f"🔧 Applying patch (Transformers v{transformers.__version__})...")

def encode_plus_patch(self, text, *args, **kwargs):
    return self(text, *args, **kwargs)

target_classes = [PreTrainedTokenizerBase, BertTokenizer, BertTokenizerFast]
for cls in target_classes:
    if not hasattr(cls, 'encode_plus'):
        cls.encode_plus = encode_plus_patch
        print(f"  ✅ Patched {cls.__name__}.encode_plus")

if not hasattr(BertTokenizer, 'build_inputs_with_special_tokens'):
    def build_inputs_patch(self, token_ids_0, token_ids_1=None):
        if token_ids_1 is None:
            return [self.cls_token_id] + token_ids_0 + [self.sep_token_id]
        return [self.cls_token_id] + token_ids_0 + [self.sep_token_id] + token_ids_1 + [self.sep_token_id]
    BertTokenizer.build_inputs_with_special_tokens = build_inputs_patch
    print(f"  ✅ Patched BertTokenizer.build_inputs_with_special_tokens")

print("✅ Compatibility patch applied.\n")

# ==========================================
# 📊 RadGraph Evaluation
# ==========================================
os.environ["CUDA_VISIBLE_DEVICES"] = ""
torch.cuda.reset_peak_memory_stats()

try:
    csv_path = "/content/drive/MyDrive/medgamma/student_distilled_reports_v2_233.csv"
    if not os.path.exists(csv_path):
        print(f"❌ File not found: {csv_path}")
    else:
        df_sub = pd.read_csv(csv_path)
        print(f"📂 Loaded: {len(df_sub)} samples")

        refs = df_sub["Ground_Truth"].fillna("").tolist()
        hyps = df_sub["Generated_Report"].fillna("").tolist()

        valid_pairs = [(h, r) for h, r in zip(hyps, refs) if h and r and len(h.strip()) > 0]
        hyps_clean, refs_clean = zip(*valid_pairs)
        print(f"✅ Valid samples: {len(valid_pairs)}")

        print("🚀 Computing RadGraph F1...")
        f1radgraph = F1RadGraph(reward_level="all")
        results = f1radgraph(hyps=list(hyps_clean), refs=list(refs_clean))

        avg_scores  = results[0]
        simple_f1   = float(avg_scores[0])
        partial_f1  = float(avg_scores[1])
        complete_f1 = float(avg_scores[2])

        print("\n" + "=" * 60)
        print("RadGraph F1 Evaluation Results (Student Model v2)")
        print("-" * 60)
        print(f"RG_E:      {complete_f1*100:.2f}%")
        print(f"RG_ER:     {partial_f1*100:.2f}%  ← commonly used in papers")
        print(f"RG_ER_bar: {simple_f1*100:.2f}%")
        print("=" * 60)

finally:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)

🔧 Applying patch (Transformers v5.2.0)...
✅ Compatibility patch applied.

📂 Loaded: 233 samples
✅ Valid samples: 233
🚀 Computing RadGraph F1...
Using device: cuda:0
model_type not provided, defaulting to radgraph-xl


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



RadGraph F1 Evaluation Results (Student Model v2)
------------------------------------------------------------
RG_E:      27.72%
RG_ER:     35.75%  ← commonly used in papers
RG_ER_bar: 37.14%


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # ⭐ 强制 RadGraph 用 CPU

import torch
import pandas as pd
from radgraph import F1RadGraph

# 重置显存统计
torch.cuda.reset_peak_memory_stats()

df_student_generated_reports = pd.read_csv(
    "/content/drive/MyDrive/medgamma/student_distilled_reports_v2_233.csv"
)
df_sub = df_student_generated_reports.copy()

refs = df_sub["Ground_Truth"].fillna("").tolist()
hyps = df_sub["Generated_Report"].fillna("").tolist()
valid_pairs = [(h, r) for h, r in zip(hyps, refs) if h and r and len(h.strip()) > 0]
hyps_clean, refs_clean = zip(*valid_pairs)

f1radgraph = F1RadGraph(reward_level="all")
results = f1radgraph(hyps=list(hyps_clean), refs=list(refs_clean))

avg_scores = results[0]
simple_f1   = float(avg_scores[0])
partial_f1  = float(avg_scores[1])
complete_f1 = float(avg_scores[2])

# 恢复 GPU 可见
os.environ.pop("CUDA_VISIBLE_DEVICES", None)

print("\n" + "=" * 60)
print("RadGraph F1 评估结果 (Student 模型 v2)")
print("-" * 60)
print(f"RG_E:      {complete_f1*100:.2f}%")
print(f"RG_ER:     {partial_f1*100:.2f}%  ← 论文常用")
print(f"RG_ER_bar: {simple_f1*100:.2f}%")
print("=" * 60)

Using device: cuda:0
model_type not provided, defaulting to radgraph-xl


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



RadGraph F1 评估结果 (Student 模型 v2)
------------------------------------------------------------
RG_E:      27.72%
RG_ER:     35.75%  ← 论文常用
RG_ER_bar: 37.14%


In [ ]:
import torch
import gc

print("=" * 50)
print("📊 GPU Memory Usage Details")
print("=" * 50)

# Total allocation
total_allocated = torch.cuda.memory_allocated(0) / (1024**3)
print(f"Total allocated: {total_allocated:.2f} GB")

# Check model objects
print("\n🔍 Large tensors in memory (>100MB):")
for obj in dir():
    try:
        if hasattr(eval(obj), 'parameters'):
            model_gb = sum(p.element_size() * p.nelement()
                          for p in eval(obj).parameters()) / (1024**3)
            if model_gb > 0.1:
                print(f"  {obj}: {model_gb:.2f} GB")
    except:
        pass

# Check all CUDA tensors
print("\n🔍 All CUDA tensors (>10MB):")
cuda_objects = []
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            size_gb = obj.element_size() * obj.nelement() / (1024**3)
            if size_gb > 0.01:
                cuda_objects.append((size_gb, obj.shape, obj.dtype))
    except:
        pass

cuda_objects.sort(reverse=True)
for size, shape, dtype in cuda_objects[:10]:
    print(f"  {size:.2f} GB  shape={shape}  dtype={dtype}")

📊 GPU Memory Usage Details
Total allocated: 11.56 GB

🔍 Large tensors in memory (>100MB):
  f1radgraph: 0.43 GB
  model: 8.01 GB
  student_model: 2.98 GB

🔍 All CUDA tensors (>10MB):
  1.25 GB  shape=torch.Size([262208, 2560])  dtype=torch.bfloat16
  1.25 GB  shape=torch.Size([262208, 2560])  dtype=torch.bfloat16
  0.09 GB  shape=torch.Size([30522, 768])  dtype=torch.float32
  0.05 GB  shape=torch.Size([10240, 2560])  dtype=torch.bfloat16
  0.05 GB  shape=torch.Size([10240, 2560])  dtype=torch.bfloat16
  0.05 GB  shape=torch.Size([10240, 2560])  dtype=torch.bfloat16
  0.05 GB  shape=torch.Size([10240, 2560])  dtype=torch.bfloat16
  0.05 GB  shape=torch.Size([10240, 2560])  dtype=torch.bfloat16
  0.05 GB  shape=torch.Size([10240, 2560])  dtype=torch.bfloat16
  0.05 GB  shape=torch.Size([10240, 2560])  dtype=torch.bfloat16


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


## 附录：RadGraph Patch（如遇到兼容性问题）

如果 RadGraph 报错 `encode_plus` 相关错误，运行下面的 patch：

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoProcessor

ckpt_path        = "/content/drive/MyDrive/medgamma/distillation_checkpoints/checkpoint-177"
FINAL_MODEL_PATH = "/content/drive/MyDrive/medgamma/distilled_final_model"

# 加载 base model + 合并 LoRA
model = AutoPeftModelForCausalLM.from_pretrained(
    ckpt_path,
    torch_dtype="auto",
    device_map="auto"
)
model = model.merge_and_unload()  # 合并 LoRA 进 base model

# processor 从原始模型加载
proc = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it")

# 保存
model.save_pretrained(FINAL_MODEL_PATH)
proc.save_pretrained(FINAL_MODEL_PATH)
print(f"✅ 合并保存完成 → {FINAL_MODEL_PATH}")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig
import torch

ckpt_path = "/content/drive/MyDrive/medgamma/distillation_checkpoints/checkpoint-177"

# 4bit 量化配置
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 加载量化 base model
base_model = AutoModelForCausalLM.from_pretrained(
    "google/medgemma-1.5-4b-it",
    quantization_config=bnb_config,
    device_map="auto"
)

# 套上 LoRA adapter
student_model = PeftModel.from_pretrained(base_model, ckpt_path)
processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it")

print("✅ 加载成功")